# Super TikTakToe: a small network with many games, on Google Colab

This trains the network **`small_many`** from scratch: the size of `small` (32 channels, 4 blocks, 0.19 million
weights), but with **2048 games per iteration, all at the same time**, and the search tree on all processor
cores. On the L4 this plays about 4.8 times as fast per move as large. The idea: a simpler network, many more games.

Schedule (by games, like large's run): 200 simulations per move until iteration 25 (about 51,000 games, like
large's iteration 100), then 400; learning rate 0.001, 0.0003 from iteration 76, 0.0001 from 126. 150
iterations are about 307,000 games, 2.5 times large's whole run. Every 4th iteration it plays 256 games
against medium (100 simulations), as large did.

For comparison, large scored against medium (by games): 51,000 games 48%; 61,000 games 49%; 82,000 games
51%; 96,000 games 55%; 123,000 games (the end) 54%. Medium itself is level with large.

Time (guessed): 4 to 6 hours, 8 to 12 compute units. The first lines show the real time per iteration.

1. `alphazero_medium.pt` must be in `Super-TikTakToe-models` in your Google Drive (it is).
2. Menu **Runtime > Change runtime type**: choose the **L4**.
3. Run the cells from top to bottom. The first line must read `Settings of 'small_many': ...`, the second
   `New 'small_many' network (0.19 million parameters) on cuda`.
4. It stops by itself after iteration 150. After a disconnect run all cells again: it continues.
5. When it has ended: **Runtime > Disconnect and delete runtime**.


In [ ]:
# Google Drive: the folder with the networks
from google.colab import drive
drive.mount('/content/drive')
MODELS = '/content/drive/MyDrive/Super-TikTakToe-models'
import os
os.environ['SUPER_TTT_MODELS'] = MODELS


In [ ]:
# The code from GitHub (the newest version every time)
%cd /content
!test -d Super-TikTakToe || git clone https://github.com/kasparhui/Super-TikTakToe.git
%cd /content/Super-TikTakToe
!git pull


In [ ]:
# Which graphics card did this session get? (must be the L4)
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# The training. Stop with the stop button of this cell: everything is saved, and running it again continues.
!python -u src/train.py run small_many 150


In [ ]:
# The checks against medium next to large's at the same number of games
import pandas as pd
new = pd.read_csv(f'{MODELS}/alphazero_small_many_log.csv')
old = pd.read_csv(f'{MODELS}/alphazero_large_log.csv')
new = new[new['vs_medium'].notna()][['iteration', 'games', 'vs_medium', 'seconds']]
old = old[old['vs_medium'].notna()][['games', 'vs_medium']].rename(columns={'vs_medium': 'large_at_same_games'})
pd.merge_asof(new.sort_values('games'), old.sort_values('games'), on='games', direction='nearest')


## After the training: direct match against medium and large

500 and 2000 simulations, 256 games per pair and setting. Results go to `alphazero_small_many_tournament.csv`.


In [ ]:
!python -u -c "import sys; sys.path.insert(0, 'src'); import test_tournament as T; T.MODEL, T.ITERATIONS, T.NEWEST, T.ALSO, T.OPEN_CHART = 'small_many', (), True, ['medium', 'large'], False; T.run(); T.summary()"
